# Limpieza Final.
## Grupo 2

### Importación de librerías

In [284]:
import pandas as pd
import numpy as np

### Importación de datos.

In [285]:
df = pd.read_csv('DATOS_CRUDOS.csv')
df.head(2)

,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0137-46,16-006,ALTA VERAPAZ,COBAN,INSTITUTO MIXTO NOCTURNO FRANCISCO MARROQUIN,6A. AVENIDA 1-15 ZONA 4,NaN,JORGE EDUARDO PAQUE LAZARO,--,DIVERSIFICADO,PRIVADO,URBANA,CERRADA DEFINITIVAMENTE,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),ALTA VERAPAZ
1,16-01-0138-46,16-01-0926,ALTA VERAPAZ,COBAN,COLEGIO COBAN,KM.2 SALIDA A SAN JUAN CHAMELCO ZONA 8,77945104,JOSE ARTURO CHOC CHEN,GUSTAVO ADOLFO SIERRA POP,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


### Eliminación de filas vacías.

In [286]:
df = df.dropna(how = 'all')

### Cambios en columna DISTRITO.

In [287]:
df['DISTRITO'] = df['DISTRITO'].fillna('NO_ESPECIFICADO')

### Cambios en columna ESTABLECIMIENTO.

In [288]:
# set(pd.Series([x for x in df['ESTABLECIMIENTO'] if len(str(x)) <= 8]))
df['ESTABLECIMIENTO'] = df['ESTABLECIMIENTO'].fillna('NO_ESPECIFICADO')

### Cambios en columna DIRECCION.

In [289]:
df['DIRECCION'] = df['DIRECCION'].str.replace(r"[0-9.,-]|^X{1,10}$|S/D|//|^\s{1,4}|\s{1,4}$", '', regex=True)
df['DIRECCION'] = df['DIRECCION'].replace({'': 'NO_ESPECIFICADO'})
quitar = list(set([x for x in df['DIRECCION'] if len(str(x)) <= 5]))
for x in ['COBAN', 'OLOPA']:
    quitar.remove(x)
df['DIRECCION'] = [np.nan if x in quitar else x for x in df['DIRECCION']]
df['DIRECCION'] = df['DIRECCION'].fillna('NO_ESPECIFICADO')

### Cambios en columna TELEFONO.

In [290]:
df['TELEFONO'] = [x if len(str(x)) >= 8 else 'NO_VALIDO' for x in df['TELEFONO']]
df['TELEFONO'] = df['TELEFONO'].str.replace(r"-|,|/", ' ', regex=True)
df['TELEFONO'] = df['TELEFONO'].str.replace(r'(\d{8})(?=\d)', r' ', regex=True)
df['TELEFONO'] = df['TELEFONO'].str.split(' ')
df['TELEFONO'] = [[y for y in x if (len(str(y)) >= 6 or y == 'NO_VALIDO')] for x in df['TELEFONO']]

### Cambios en columna SUPERVISOR.

In [291]:
df['SUPERVISOR'] = df['SUPERVISOR'].fillna('NO_ESPECIFICADO')

### Cambios en columna DIRECTOR.

In [292]:
df['DIRECTOR'] = df['DIRECTOR'].str.replace(r"[0-9.,-]|^X{1,10}$|S/D|^\s", '', regex=True)
df['DIRECTOR'] = df['DIRECTOR'].fillna('NO_ESPECIFICADO')

### Verificando nulos.

In [293]:
n_registros, _ = df.shape
faltantes = {'Cantidad': [],
             'Porcentaje %': []}

for x in df.columns:
    cantidad = int(df[x].isna().sum())
    porcentaje = cantidad*100/n_registros

    faltantes['Cantidad'].append(cantidad)
    faltantes['Porcentaje %'].append(porcentaje)

faltantes_df = pd.DataFrame(faltantes, index= df.columns)
faltantes_df[faltantes_df['Cantidad'] > 0]

,Cantidad,Porcentaje %


### Eliminación de columna DEPARTAMENTAL.

In [294]:
df = df.drop(columns=['DEPARTAMENTAL'])

### Normas ortográficas.

In [295]:
tabla = str.maketrans("ÁÉÍÓÚÜáéíóúü", "AEIOUUAEIOUU")
columnas_texto = df.select_dtypes(include=['object', 'string']).columns
df[columnas_texto] = df[columnas_texto].apply(lambda columna: (columna.astype("string").str.upper().str.translate(tabla).str.strip()))

### Exportación de datos.

In [296]:
df.to_csv('DARTOS_LIMPIOS.csv')